checks the file list

In [1]:
from huggingface_hub import HfApi

files = HfApi().list_repo_files(
    'fever/fever',
    repo_type='dataset',
    revision='refs/convert/parquet'
)

print("TRAIN AND VALIDATION FILES:")
print([
    f for f in files
    if f.startswith('v1.0/train')
    or f.startswith('v1.0/paper_dev')
])

print("\nWIKIPEDIA FILES:")
print([
    f for f in files
    if f.startswith('wiki_pages/')
])

TRAIN AND VALIDATION FILES:
['v1.0/paper_dev/0000.parquet', 'v1.0/train/0000.parquet']

WIKIPEDIA FILES:
['wiki_pages/partial-wikipedia_pages/0000.parquet', 'wiki_pages/partial-wikipedia_pages/0001.parquet', 'wiki_pages/partial-wikipedia_pages/0002.parquet', 'wiki_pages/partial-wikipedia_pages/0003.parquet', 'wiki_pages/partial-wikipedia_pages/0004.parquet', 'wiki_pages/partial-wikipedia_pages/0005.parquet', 'wiki_pages/partial-wikipedia_pages/0006.parquet', 'wiki_pages/partial-wikipedia_pages/0007.parquet', 'wiki_pages/partial-wikipedia_pages/0008.parquet', 'wiki_pages/partial-wikipedia_pages/0009.parquet']


Load FEVER datasets

In [2]:
import pandas as pd

train_path = "hf://datasets/fever/fever@refs/convert/parquet/v1.0/train/0000.parquet"
val_path = "hf://datasets/fever/fever@refs/convert/parquet/v1.0/paper_dev/0000.parquet"

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

print("TRAIN DATASET")
print("Shape:", train_df.shape)
display(train_df.head())

print("\nVALIDATION DATASET")
print("Shape:", val_df.shape)
display(val_df.head())

TRAIN DATASET
Shape: (311431, 7)


,id,label,claim,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Nikolaj_Coster-Waldau,7
1,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Fox_Broadcasting_Company,-1
2,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187498,Roman_Atwood,1
3,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187499,Roman_Atwood,3
4,214861,SUPPORTS,"History of art includes architecture, dance, s...",255136,254645,History_of_art,2



VALIDATION DATASET
Shape: (18999, 7)


,id,label,claim,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,91198,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback...,108548,-1,,-1
1,194462,NOT ENOUGH INFO,Tilda Swinton is a vegan.,227768,-1,,-1
2,137334,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,289914,283015,Soul_Food_-LRB-film-RRB-,-1
3,137334,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,291259,284217,Soul_Food_-LRB-film-RRB-,-1
4,137334,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,293412,285960,Soul_Food_-LRB-film-RRB-,-1


In [3]:
# Check column data types
print("TRAIN DATA TYPES:")
print(train_df.dtypes)

print("\nVALIDATION DATA TYPES:")
print(val_df.dtypes)


# Count total rows and unique claims
print("\nTRAIN DATASET COUNTS")
print("Total rows:", len(train_df))
print("Unique claims:", train_df["id"].nunique())

print("\nVALIDATION DATASET COUNTS")
print("Total rows:", len(val_df))
print("Unique claims:", val_df["id"].nunique())


# Check how many rows each claim occupies
train_rows_per_claim = train_df.groupby("id").size()
val_rows_per_claim = val_df.groupby("id").size()

print("\nTRAIN ROWS PER CLAIM")
print(train_rows_per_claim.describe())

print("\nVALIDATION ROWS PER CLAIM")
print(val_rows_per_claim.describe())

TRAIN DATA TYPES:
id                         int32
label                     object
claim                     object
evidence_annotation_id     int32
evidence_id                int32
evidence_wiki_url         object
evidence_sentence_id       int32
dtype: object

VALIDATION DATA TYPES:
id                         int32
label                     object
claim                     object
evidence_annotation_id     int32
evidence_id                int32
evidence_wiki_url         object
evidence_sentence_id       int32
dtype: object

TRAIN DATASET COUNTS
Total rows: 311431
Unique claims: 145449

VALIDATION DATASET COUNTS
Total rows: 18999
Unique claims: 9999

TRAIN ROWS PER CLAIM
count    145449.000000
mean          2.141170
std           3.663248
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         338.000000
dtype: float64

VALIDATION ROWS PER CLAIM
count    9999.000000
mean        1.900090
std         2.778356
min         1.000000
25%     

Inspect the label distribution

In [4]:
# Count labels using unique claims only

train_labels = train_df.drop_duplicates("id")["label"].value_counts()
val_labels = val_df.drop_duplicates("id")["label"].value_counts()

print("TRAIN LABEL DISTRIBUTION")
print(train_labels)

print("\nTRAIN LABEL PERCENTAGES")
print((train_labels / train_labels.sum() * 100).round(2))


print("\nVALIDATION LABEL DISTRIBUTION")
print(val_labels)

print("\nVALIDATION LABEL PERCENTAGES")
print((val_labels / val_labels.sum() * 100).round(2))

TRAIN LABEL DISTRIBUTION
label
SUPPORTS           80035
NOT ENOUGH INFO    35639
REFUTES            29775
Name: count, dtype: int64

TRAIN LABEL PERCENTAGES
label
SUPPORTS           55.03
NOT ENOUGH INFO    24.50
REFUTES            20.47
Name: count, dtype: float64

VALIDATION LABEL DISTRIBUTION
label
NOT ENOUGH INFO    3333
SUPPORTS           3333
REFUTES            3333
Name: count, dtype: int64

VALIDATION LABEL PERCENTAGES
label
NOT ENOUGH INFO    33.33
SUPPORTS           33.33
REFUTES            33.33
Name: count, dtype: float64


Check missing values

In [5]:
print("TRAIN MISSING VALUES")
print(train_df.isnull().sum())

print("\nVALIDATION MISSING VALUES")
print(val_df.isnull().sum())

TRAIN MISSING VALUES
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
dtype: int64

VALIDATION MISSING VALUES
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
dtype: int64


check duplicate

In [6]:
print("TRAIN EXACT DUPLICATE ROWS:")
print(train_df.duplicated().sum())

print("\nVALIDATION EXACT DUPLICATE ROWS:")
print(val_df.duplicated().sum())

TRAIN EXACT DUPLICATE ROWS:
6

VALIDATION EXACT DUPLICATE ROWS:
0


Check duplicate claim text under different IDs

In [7]:
train_unique = train_df.drop_duplicates("id")
val_unique = val_df.drop_duplicates("id")

print("TRAIN CLAIM TEXT DUPLICATES:")
print(train_unique["claim"].duplicated().sum())

print("\nVALIDATION CLAIM TEXT DUPLICATES:")
print(val_unique["claim"].duplicated().sum())

TRAIN CLAIM TEXT DUPLICATES:
9904

VALIDATION CLAIM TEXT DUPLICATES:
256


Check claims with multiple labels

In [8]:
train_label_counts = train_df.groupby("id")["label"].nunique()
val_label_counts = val_df.groupby("id")["label"].nunique()

print("TRAIN CLAIMS WITH MULTIPLE LABELS:")
print((train_label_counts > 1).sum())

print("\nVALIDATION CLAIMS WITH MULTIPLE LABELS:")
print((val_label_counts > 1).sum())

TRAIN CLAIMS WITH MULTIPLE LABELS:
0

VALIDATION CLAIMS WITH MULTIPLE LABELS:
0


Check NEI claims with Wikipedia page references

In [9]:
print("TRAIN NEI ROWS WITH PAGE POINTERS:")
print(
    (
        (train_df["label"] == "NOT ENOUGH INFO") &
        (train_df["evidence_wiki_url"] != "")
    ).sum()
)

print("\nVALIDATION NEI ROWS WITH PAGE POINTERS:")
print(
    (
        (val_df["label"] == "NOT ENOUGH INFO") &
        (val_df["evidence_wiki_url"] != "")
    ).sum()
)

TRAIN NEI ROWS WITH PAGE POINTERS:
0

VALIDATION NEI ROWS WITH PAGE POINTERS:
0


Check label distribution by unique claim ID

In [10]:
print("TRAIN LABEL DISTRIBUTION (UNIQUE CLAIM IDs):")
print(train_unique["label"].value_counts())

print("\nVALIDATION LABEL DISTRIBUTION (UNIQUE CLAIM IDs):")
print(val_unique["label"].value_counts())

TRAIN LABEL DISTRIBUTION (UNIQUE CLAIM IDs):
label
SUPPORTS           80035
NOT ENOUGH INFO    35639
REFUTES            29775
Name: count, dtype: int64

VALIDATION LABEL DISTRIBUTION (UNIQUE CLAIM IDs):
label
NOT ENOUGH INFO    3333
SUPPORTS           3333
REFUTES            3333
Name: count, dtype: int64


Generates inspection summary

In [11]:
print("=" * 50)
print("FEVER DATASET INSPECTION SUMMARY")
print("=" * 50)

for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df)
]:
    print(f"\n{name} DATASET")
    print("-" * 30)

    print("Total rows:", len(df))
    print("Unique claims:", df["id"].nunique())
    print("Duplicate rows:", df.duplicated().sum())

    unique_claims = df.drop_duplicates("id")

    print(
        "Duplicate claim texts:",
        unique_claims["claim"].duplicated().sum()
    )

    print(
        "Claims with multiple labels:",
        (df.groupby("id")["label"].nunique() > 1).sum()
    )

    print("\nLabel distribution:")
    print(
        unique_claims["label"].value_counts()
    )

    print("\nMissing values:")
    print(df.isnull().sum())

FEVER DATASET INSPECTION SUMMARY

TRAIN DATASET
------------------------------
Total rows: 311431
Unique claims: 145449
Duplicate rows: 6
Duplicate claim texts: 9904
Claims with multiple labels: 0

Label distribution:
label
SUPPORTS           80035
NOT ENOUGH INFO    35639
REFUTES            29775
Name: count, dtype: int64

Missing values:
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
dtype: int64

VALIDATION DATASET
------------------------------
Total rows: 18999
Unique claims: 9999
Duplicate rows: 0
Duplicate claim texts: 256
Claims with multiple labels: 0

Label distribution:
label
NOT ENOUGH INFO    3333
SUPPORTS           3333
REFUTES            3333
Name: count, dtype: int64

Missing values:
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
ev

In [14]:
for name, d in (("TRAIN", train_df), ("VALIDATION", val_df)):
    sr  = d[d["label"] != "NOT ENOUGH INFO"]
    nei = d[d["label"] == "NOT ENOUGH INFO"]
    print(f"\n{name}")
    print("  SUPPORTS/REFUTES rows:", len(sr),
          "| empty page:", int((sr["evidence_wiki_url"] == "").sum()),
          "| sentence_id == -1:", int((sr["evidence_sentence_id"] == -1).sum()))
    print("  NEI rows:", len(nei), "| NEI claims:", nei["id"].nunique(),
          "| NEI rows with empty page:", int((nei["evidence_wiki_url"] == "").sum()))

    u = d.drop_duplicates("id")
    ids_per_text = u.groupby("claim")["id"].nunique()
    labels_per_text = u.groupby("claim")["label"].nunique()
    print("  claim texts used by more than one ID:", int((ids_per_text > 1).sum()),
          "| of those with conflicting labels:", int((labels_per_text > 1).sum()))

tr_texts = set(train_df["claim"])
va_u = val_df.drop_duplicates("id")
print("\nValidation claims whose exact text also appears in train:",
      int(va_u["claim"].isin(tr_texts).sum()), "of", len(va_u))


TRAIN
  SUPPORTS/REFUTES rows: 263822 | empty page: 0 | sentence_id == -1: 95529
  NEI rows: 47609 | NEI claims: 35639 | NEI rows with empty page: 47609
  claim texts used by more than one ID: 5894 | of those with conflicting labels: 753

VALIDATION
  SUPPORTS/REFUTES rows: 14475 | empty page: 0 | sentence_id == -1: 6279
  NEI rows: 4524 | NEI claims: 3333 | NEI rows with empty page: 4524
  claim texts used by more than one ID: 206 | of those with conflicting labels: 32

Validation claims whose exact text also appears in train: 1 of 9999
